# FMGAI Group Assignment, Phase 2: noise-schedule ablation on InstructPix2Pix

4× aerial super-resolution on UC Merced. Run 0 = pretrained, Run A = original schedule {β_t}, Run B = {β_t^x}, x = 1.60.

**Setup:** upload the `phase2` folder to Google Drive at `MyDrive/FMGAI_A1/phase2`, then choose *Runtime → Change runtime type → GPU*.
Results go to `MyDrive/FMGAI_A1/results`. Each training run takes about 1.5–3 h on a T4. If the session dies, rerun the setup
cells (1–3) and then the same training cell. It resumes from the last saved state.

In [ ]:
# 1. Mount Drive and go to the code folder
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/FMGAI_A1/phase2'
OUT = '/content/drive/MyDrive/FMGAI_A1/results'
DATA = '/content/data'          # rebuilt locally each session (deterministic, about 2 min); faster than reading from Drive
import os
os.makedirs(OUT, exist_ok=True)
%cd {PROJ}
!nvidia-smi

In [ ]:
# 2. Install and record the exact environment for the README
!pip install -q -r requirements.txt
!python -c "import sys, torch, diffusers, transformers, accelerate, datasets, bitsandbytes, lpips, skimage; print('python', sys.version.split()[0]); print('torch', torch.__version__, 'cuda', torch.version.cuda); print('gpu', torch.cuda.get_device_name(0)); [print(m.__name__, m.__version__) for m in (diffusers, transformers, accelerate, datasets, bitsandbytes, skimage)]; print('lpips', getattr(lpips, '__version__', 'n/a'))" | tee {OUT}/environment.txt

In [ ]:
# 3. Build the LR/HR pairs and the 1470/315/315 split
!python prepare_data.py --data_dir {DATA}
!cp {DATA}/splits.csv {DATA}/dataset_summary.json {OUT}/

In [ ]:
# 4. Schedule Analysis figures and the original-vs-modified code listing
!python plot_schedule.py --out_dir {OUT}/figures
!python make_diff.py --out_dir {OUT}/figures

### Optional smoke test (about 5 min)
One short epoch on 16 images to check the whole loop (CSV, validation, samples, saving) before the long runs. Delete `smoke/` afterwards.

In [ ]:
!python train_instruct_pix2pix.py --noise_schedule power --output_dir /content/smoke   --pretrained_model_name_or_path timbrooks/instruct-pix2pix --data_dir {DATA} --max_train_samples 16   --resolution 256 --random_flip --train_batch_size 4 --gradient_accumulation_steps 2 --num_train_epochs 1   --learning_rate 5e-5 --lr_scheduler constant --lr_warmup_steps 0 --conditioning_dropout_prob 0.05   --mixed_precision fp16 --use_8bit_adam --gradient_checkpointing --num_val_metric_images 8 --seed 42
!cat /content/smoke/train_log.csv /content/smoke/schedule_check.json && ls /content/smoke/samples

In [ ]:
# 5. Run 0 (pretrained, no fine-tuning) and the bicubic reference on the test split
!python infer.py --run bicubic --data_dir {DATA} --out_dir {OUT}/eval
!python infer.py --run run0 --data_dir {DATA} --out_dir {OUT}/eval

In [ ]:
# 6. Shared fine-tuning arguments: identical for Run A and Run B
COMMON = ('--pretrained_model_name_or_path timbrooks/instruct-pix2pix --data_dir ' + DATA +
          ' --resolution 256 --random_flip --train_batch_size 4 --gradient_accumulation_steps 2'
          ' --num_train_epochs 10 --learning_rate 5e-5 --lr_scheduler constant --lr_warmup_steps 0'
          ' --adam_weight_decay 1e-2 --max_grad_norm 1.0 --conditioning_dropout_prob 0.05'
          ' --mixed_precision fp16 --use_8bit_adam --gradient_checkpointing'
          ' --checkpointing_steps 368 --checkpoints_total_limit 1 --seed 42 --resume_from_checkpoint latest')
print(COMMON)

In [ ]:
# 7. Run A: original schedule {beta_t}
!python train_instruct_pix2pix.py --noise_schedule original --output_dir {OUT}/outputs/runA {COMMON}

In [ ]:
# 8. Run B: modified schedule {beta_t ** 1.60}
!python train_instruct_pix2pix.py --noise_schedule power --output_dir {OUT}/outputs/runB {COMMON}

In [ ]:
# 9. Test-set scores and inference trajectories of the final models
!python infer.py --run runA --data_dir {DATA} --out_dir {OUT}/eval --model_dir {OUT}/outputs/runA
!python infer.py --run runB --data_dir {DATA} --out_dir {OUT}/eval --model_dir {OUT}/outputs/runB

In [ ]:
# 10. Report figures and the results table
!python make_figures.py --runs_dir {OUT}/outputs --eval_dir {OUT}/eval --data_dir {DATA} --out_dir {OUT}/figures
from IPython.display import Image, display, Markdown
display(Markdown(open(f'{OUT}/figures/results_table.md').read()))
for f in ['loss_curves.png', 'checkpoints_runA.png', 'checkpoints_runB.png', 'test_comparison.png']:
    display(Image(f'{OUT}/figures/{f}'))
display(Image(f'{OUT}/eval/runA/trajectories/runA_trajectory.png'))
display(Image(f'{OUT}/eval/runB/trajectories/runB_trajectory.png'))

In [ ]:
# 11. Submission zip: code, logs, samples, trajectories, figures (no model weights). Add the report PDF before submitting.
import shutil, glob, os
pack = '/content/submission'
shutil.rmtree(pack, ignore_errors=True)
shutil.copytree(PROJ, f'{pack}/code', ignore=shutil.ignore_patterns('data', 'figures', '__pycache__', '*.ipynb_checkpoints'))
for run in ['runA', 'runB']:
    d = f'{OUT}/outputs/{run}'
    os.makedirs(f'{pack}/logs/{run}', exist_ok=True)
    for f in ['train_log.csv', 'val_log.csv', 'val_metrics.csv', 'schedule.json', 'schedule_check.json']:
        if os.path.exists(f'{d}/{f}'): shutil.copy(f'{d}/{f}', f'{pack}/logs/{run}/')
    shutil.copytree(f'{d}/samples', f'{pack}/checkpoint_samples/{run}')
for run in ['bicubic', 'run0', 'runA', 'runB']:
    d = f'{OUT}/eval/{run}'
    os.makedirs(f'{pack}/eval/{run}', exist_ok=True)
    for f in ['summary.json', 'per_image_metrics.csv']:
        shutil.copy(f'{d}/{f}', f'{pack}/eval/{run}/')
    if os.path.isdir(f'{d}/trajectories'): shutil.copytree(f'{d}/trajectories', f'{pack}/trajectories/{run}')
shutil.copytree(f'{OUT}/figures', f'{pack}/figures')
for f in ['environment.txt', 'splits.csv', 'dataset_summary.json']: shutil.copy(f'{OUT}/{f}', pack)
shutil.make_archive(f'{OUT}/FMGAI_A1_phase2_submission', 'zip', pack)
print('wrote', f'{OUT}/FMGAI_A1_phase2_submission.zip')